# Aula 02 - Persistência de Arquivos: CSV, TSV e Planilhas
**QXD0099 - Desenvolvimento de Software para Persistência**  
Universidade Federal do Ceará - Campus Quixadá  
Prof. Francisco Victor da Silva Pinheiro  
victorpinheiro@ufc.br

> Notebook prático e comentado para execução em sala no Google Colab.

## Agenda

Nesta aula vamos trabalhar com:

- CSV — Comma-Separated Values;
- padrão RFC 4180;
- problemas comuns em arquivos CSV;
- leitura e escrita de CSV;
- TSV — Tab-Separated Values;
- planilhas Excel;
- diferenças entre CSV, TSV e XLSX;
- módulo nativo `csv`;
- `pandas`;
- `openpyxl`;
- histogramas;
- boxplots;
- gráficos de pizza;
- análise de frequência e desempenho dos alunos;
- exportação para Excel;
- leitura eficiente de arquivos maiores.

# 1. CSV — Comma-Separated Values

Cada linha representa um registro e os campos são separados por vírgulas.

Exemplo:

```text
Nome,Idade,Cidade
João da Silva,30,Fortaleza
Maria Lopes,28,São Paulo
```

CSV é um formato textual simples e amplamente suportado.

## 2. Criando um CSV simples manualmente

In [ ]:
# Vamos criar um arquivo CSV utilizando escrita de texto comum.
with open("alunos_simples.csv", "w", encoding="utf-8") as arquivo:
    arquivo.write("Nome,Idade,Cidade\n")
    arquivo.write("João da Silva,30,Fortaleza\n")
    arquivo.write("Maria Lopes,28,São Paulo\n")

print("Arquivo alunos_simples.csv criado.")

Arquivo alunos_simples.csv criado.


In [ ]:
# Visualizando o conteúdo persistido.
with open("alunos_simples.csv", "r", encoding="utf-8") as arquivo:
    print(arquivo.read())

Nome,Idade,Cidade
João da Silva,30,Fortaleza
Maria Lopes,28,São Paulo



# 3. RFC 4180 e campos especiais

Campos contendo:

- vírgulas;
- aspas;
- quebras de linha;

devem ser tratados corretamente.

Exemplo válido:

```text
"Nome","Idade","Cidade"
"João da Silva",30,"Fortaleza"
"Maria, Lopes",28,"São Paulo"
```

## 4. Criando um CSV com casos especiais usando `csv`

In [ ]:
import csv

# newline="" é recomendado ao trabalhar com o módulo csv,
# principalmente para evitar linhas em branco extras em alguns sistemas.
with open("pessoas_rfc.csv", "w", newline="", encoding="utf-8") as arquivo:

    writer = csv.writer(arquivo)

    writer.writerow(["Nome", "Idade", "Cidade", "Comentário"])
    writer.writerow(["João da Silva", 30, "Fortaleza", "Aluno participativo"])
    writer.writerow(["Maria, Lopes", 28, "São Paulo, SP", 'Disse: "Gostei muito!"'])
    writer.writerow(["Lucas", 22, "Quixadá", "Aluno excelente\nParticipativo em aula"])

print("CSV criado.")

CSV criado.


In [ ]:
# Vamos visualizar o texto bruto para observar
# como aspas, vírgulas e quebras de linha foram tratadas.
with open("pessoas_rfc.csv", "r", encoding="utf-8") as arquivo:
    print(arquivo.read())

Nome,Idade,Cidade,Comentário
João da Silva,30,Fortaleza,Aluno participativo
"Maria, Lopes",28,"São Paulo, SP","Disse: ""Gostei muito!"""
Lucas,22,Quixadá,"Aluno excelente
Participativo em aula"



## 5. Por que usar a biblioteca `csv`?

Montar CSV manualmente com `split(",")` pode falhar quando um campo contém vírgulas ou aspas.

A biblioteca `csv` conhece as regras do formato e trata esses casos.

In [ ]:
import csv

with open("pessoas_rfc.csv", "r", newline="", encoding="utf-8") as arquivo:

    reader = csv.reader(arquivo)

    for linha in reader:
        print(linha)

['Nome', 'Idade', 'Cidade', 'Comentário']
['João da Silva', '30', 'Fortaleza', 'Aluno participativo']
['Maria, Lopes', '28', 'São Paulo, SP', 'Disse: "Gostei muito!"']
['Lucas', '22', 'Quixadá', 'Aluno excelente\nParticipativo em aula']


## 6. `csv.DictWriter`

In [ ]:
import csv

alunos = [
    {"Nome": "Ana", "Nota1": 8.5, "Nota2": 9.0},
    {"Nome": "Bruno", "Nota1": 6.0, "Nota2": 7.5},
    {"Nome": "Carlos", "Nota1": 9.2, "Nota2": 8.8}
]

with open("notas_dict.csv", "w", newline="", encoding="utf-8") as arquivo:

    campos = ["Nome", "Nota1", "Nota2"]

    writer = csv.DictWriter(arquivo, fieldnames=campos)

    # Escreve o cabeçalho.
    writer.writeheader()

    # Escreve todos os registros.
    writer.writerows(alunos)

print("notas_dict.csv criado.")

notas_dict.csv criado.


## 7. `csv.DictReader`

In [ ]:
import csv

with open("notas_dict.csv", "r", newline="", encoding="utf-8") as arquivo:

    reader = csv.DictReader(arquivo)

    for aluno in reader:
        print(aluno["Nome"], aluno["Nota1"], aluno["Nota2"])

Ana 8.5 9.0
Bruno 6.0 7.5
Carlos 9.2 8.8


# 8. Abrindo CSV com pandas

O `pandas` trabalha com estruturas tabulares chamadas **DataFrames**.

In [ ]:
import pandas as pd

df = pd.read_csv("notas_dict.csv")

df

,Nome,Nota1,Nota2
0,Ana,8.5,9.0
1,Bruno,6.0,7.5
2,Carlos,9.2,8.8


### Informações sobre o DataFrame

In [ ]:
print("Linhas e colunas:", df.shape)
print("\nColunas:")
print(df.columns.tolist())

print("\nTipos:")
print(df.dtypes)

Linhas e colunas: (3, 3)

Colunas:
['Nome', 'Nota1', 'Nota2']

Tipos:
Nome      object
Nota1    float64
Nota2    float64
dtype: object


### Selecionando colunas

In [ ]:
# Uma coluna de um DataFrame pode ser acessada pelo nome.
print(df["Nome"])

print("\nNotas:")
print(df[["Nome", "Nota1", "Nota2"]])

0       Ana
1     Bruno
2    Carlos
Name: Nome, dtype: object

Notas:
     Nome  Nota1  Nota2
0     Ana    8.5    9.0
1   Bruno    6.0    7.5
2  Carlos    9.2    8.8


### Criando uma coluna calculada

In [ ]:
# Criamos uma nova coluna a partir de outras colunas.
df["Media"] = (df["Nota1"] + df["Nota2"]) / 2

df

,Nome,Nota1,Nota2,Media
0,Ana,8.5,9.0,8.75
1,Bruno,6.0,7.5,6.75
2,Carlos,9.2,8.8,9.00


### Filtrando dados

In [ ]:
# Mantemos apenas alunos com média maior ou igual a 7.
aprovados = df[df["Media"] >= 7]

aprovados

,Nome,Nota1,Nota2,Media
0,Ana,8.5,9.0,8.75
2,Carlos,9.2,8.8,9.00


# 9. TSV — Tab-Separated Values

TSV utiliza **tabulação (`\t`)** como separador.

Exemplo:

```text
Nome    Idade    Cidade
Ana     20       Quixadá
```

Uma vantagem é reduzir conflitos quando os dados possuem muitas vírgulas.

## 10. Criando um TSV

In [ ]:
import csv

with open("alunos.tsv", "w", newline="", encoding="utf-8") as arquivo:

    # delimiter="\t" define TAB como separador.
    writer = csv.writer(arquivo, delimiter="\t")

    writer.writerow(["Nome", "Curso", "Nota"])
    writer.writerow(["Ana", "Computação", 8.5])
    writer.writerow(["Bruno", "Engenharia", 7.2])
    writer.writerow(["Carla", "Design", 9.1])

print("alunos.tsv criado.")

In [ ]:
# O \t aparece visualmente como uma tabulação.
with open("alunos.tsv", "r", encoding="utf-8") as arquivo:
    print(arquivo.read())

## 11. Lendo TSV com pandas

In [ ]:
import pandas as pd

# sep="\t" informa ao pandas que o separador é TAB.
df_tsv = pd.read_csv("alunos.tsv", sep="\t")

df_tsv

# 12. CSV x TSV x XLSX

| Formato | Quando usar |
|---|---|
| CSV | Dados tabulares simples e alta compatibilidade |
| TSV | Dados tabulares em que vírgulas aparecem com frequência |
| XLSX | Formatação, fórmulas, múltiplas abas e integração com Excel |

CSV e TSV são arquivos de texto.  
XLSX é um formato estruturado de planilha.

# 13. Planilhas Excel

Vamos criar um arquivo `.xlsx` para que todos os exemplos funcionem no Colab.

In [ ]:
import pandas as pd

dados_alunos = pd.DataFrame({
    "Aluno": ["Ana", "Bruno", "Carla", "Daniel"],
    "Curso": ["Computação", "Engenharia", "Design", "Computação"],
    "Nota": [8.5, 6.5, 9.2, 7.8]
})

dados_professores = pd.DataFrame({
    "Professor": ["Victor", "Maria"],
    "Disciplina": ["Persistência", "Banco de Dados"]
})

# ExcelWriter permite criar múltiplas abas no mesmo arquivo.
with pd.ExcelWriter("dados_academicos.xlsx", engine="openpyxl") as writer:

    dados_alunos.to_excel(
        writer,
        sheet_name="Alunos",
        index=False
    )

    dados_professores.to_excel(
        writer,
        sheet_name="Professores",
        index=False
    )

print("dados_academicos.xlsx criado.")

## 14. Abrindo uma planilha específica

In [ ]:
import pandas as pd

df_excel = pd.read_excel(
    "dados_academicos.xlsx",
    sheet_name="Alunos"
)

df_excel

## 15. Carregando todas as abas

In [ ]:
# sheet_name=None retorna um dicionário:
# chave = nome da aba
# valor = DataFrame
dfs = pd.read_excel(
    "dados_academicos.xlsx",
    sheet_name=None
)

print("Abas encontradas:")
print(dfs.keys())

In [ ]:
print("ABA ALUNOS")
display(dfs["Alunos"])

print("\nABA PROFESSORES")
display(dfs["Professores"])

## 16. Lendo uma aba pelo índice

In [ ]:
# sheet_name=0 significa primeira aba.
primeira_aba = pd.read_excel(
    "dados_academicos.xlsx",
    sheet_name=0
)

primeira_aba

# 17. Manipulação com `openpyxl`

`pandas` é excelente para análise tabular.

`openpyxl` permite trabalhar diretamente com elementos de uma planilha Excel, como células e abas.

In [ ]:
from openpyxl import load_workbook

# Carrega o arquivo Excel já existente.
workbook = load_workbook("dados_academicos.xlsx")

print("Abas:")
print(workbook.sheetnames)

# Selecionamos a aba Alunos.
planilha = workbook["Alunos"]

print("A1:", planilha["A1"].value)
print("A2:", planilha["A2"].value)
print("C2:", planilha["C2"].value)

## 18. Alterando uma célula

In [ ]:
from openpyxl import load_workbook

workbook = load_workbook("dados_academicos.xlsx")
planilha = workbook["Alunos"]

# Alteramos diretamente o conteúdo de uma célula.
planilha["C2"] = "Sistemas de Informação"

# Salvamos em outro arquivo para preservar o original.
workbook.save("dados_academicos_alterado.xlsx")

print("Arquivo alterado criado.")

# 19. Dataset de notas para visualização

Vamos gerar um arquivo `notas.csv` para os exemplos de histogramas e boxplots.

In [ ]:
import pandas as pd

notas = pd.DataFrame({
    "aluno": [
        "Ana", "Bruno", "Carla", "Daniel", "Eduarda",
        "Felipe", "Gabriela", "Henrique", "Isabela", "João",
        "Karen", "Lucas", "Marina", "Nicolas", "Olivia"
    ],
    "nota": [
        8.5, 6.0, 9.2, 7.5, 5.8,
        4.5, 8.0, 7.2, 9.8, 6.7,
        3.5, 8.8, 7.9, 5.0, 9.0
    ]
})

notas.to_csv("notas.csv", index=False, encoding="utf-8")

print("notas.csv criado.")

## 20. Histograma das notas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_notas = pd.read_csv("notas.csv")

plt.figure(figsize=(8, 4))

# Histograma mostra a distribuição dos valores em intervalos.
plt.hist(df_notas["nota"], bins=5)

plt.title("Distribuição das Notas")
plt.xlabel("Nota")
plt.ylabel("Frequência")
plt.grid(True)
plt.show()

### Interpretação

O histograma ajuda a observar:

- concentração de notas;
- intervalos mais frequentes;
- possíveis extremos;
- formato geral da distribuição.

## 21. Boxplot das notas

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))

# Boxplot mostra mediana, quartis, dispersão e possíveis outliers.
plt.boxplot(df_notas["nota"])

plt.title("Boxplot das Notas")
plt.ylabel("Nota")
plt.show()

# 22. Criando status dos alunos

Vamos classificar:

- **Aprovado**: média ≥ 7;
- **Reprovado**: média < 7.

In [ ]:
df_notas["status"] = df_notas["nota"].apply(
    lambda nota: "Aprovado" if nota >= 7 else "Reprovado"
)

df_notas

## 23. Contagem por status

In [ ]:
contagem = df_notas["status"].value_counts()

print(contagem)

## 24. Gráfico de pizza

In [ ]:
import matplotlib.pyplot as plt

contagem = df_notas["status"].value_counts()

plt.figure(figsize=(6, 6))

plt.pie(
    contagem,
    labels=contagem.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Distribuição de Aprovados e Reprovados")
plt.axis("equal")
plt.show()

# 25. Análise de Frequência e Desempenho dos Alunos

Vamos criar `frequencia_notas.csv`.

Cada registro terá:

- Aluno;
- Curso;
- Data;
- Presenca;
- Nota.

Quando o aluno falta, a nota será vazia (`NaN` ao ser lida pelo pandas).

In [ ]:
import pandas as pd
import math

dados = [
    ["Ana", "Computação", "01/08/2026", "Sim", 8.0],
    ["Ana", "Computação", "08/08/2026", "Sim", 9.0],
    ["Ana", "Computação", "15/08/2026", "Não", None],
    ["Ana", "Computação", "22/08/2026", "Sim", 8.5],

    ["Bruno", "Engenharia", "01/08/2026", "Sim", 6.0],
    ["Bruno", "Engenharia", "08/08/2026", "Não", None],
    ["Bruno", "Engenharia", "15/08/2026", "Sim", 5.5],
    ["Bruno", "Engenharia", "22/08/2026", "Sim", 6.5],

    ["Carla", "Design", "01/08/2026", "Sim", 9.0],
    ["Carla", "Design", "08/08/2026", "Sim", 8.5],
    ["Carla", "Design", "15/08/2026", "Sim", 9.5],
    ["Carla", "Design", "22/08/2026", "Sim", 10.0],

    ["Daniel", "Computação", "01/08/2026", "Não", None],
    ["Daniel", "Computação", "08/08/2026", "Sim", 4.0],
    ["Daniel", "Computação", "15/08/2026", "Sim", 5.0],
    ["Daniel", "Computação", "22/08/2026", "Sim", 4.5],

    ["Eduarda", "Engenharia", "01/08/2026", "Sim", 7.0],
    ["Eduarda", "Engenharia", "08/08/2026", "Sim", 7.5],
    ["Eduarda", "Engenharia", "15/08/2026", "Sim", 8.0],
    ["Eduarda", "Engenharia", "22/08/2026", "Não", None],

    ["Felipe", "Design", "01/08/2026", "Sim", 5.0],
    ["Felipe", "Design", "08/08/2026", "Sim", 6.0],
    ["Felipe", "Design", "15/08/2026", "Sim", 5.5],
    ["Felipe", "Design", "22/08/2026", "Sim", 5.8],
]

df_frequencia = pd.DataFrame(
    dados,
    columns=["Aluno", "Curso", "Data", "Presenca", "Nota"]
)

df_frequencia.to_csv(
    "frequencia_notas.csv",
    index=False,
    encoding="utf-8"
)

print("frequencia_notas.csv criado.")

df_frequencia

## 26. Leitura dos dados

In [ ]:
import pandas as pd

df = pd.read_csv("frequencia_notas.csv")

df

## 27. Verificando tipos e valores ausentes

In [ ]:
print(df.dtypes)

print("\nValores ausentes por coluna:")
print(df.isna().sum())

## 28. Convertendo a coluna Data

In [ ]:
# dayfirst=True indica que o formato é dia/mês/ano.
df["Data"] = pd.to_datetime(
    df["Data"],
    format="%d/%m/%Y"
)

print(df.dtypes)

df.head()

# 29. Filtrando presenças com nota válida

Conforme o slide, usamos apenas registros:

- com `Presenca == "Sim"`;
- com nota não nula.

In [ ]:
df_presente = df[
    (df["Presenca"] == "Sim") &
    (df["Nota"].notna())
]

df_presente

## 30. Média por aluno

In [ ]:
medias_alunos = (
    df_presente
    .groupby("Aluno", as_index=False)["Nota"]
    .mean()
)

medias_alunos.rename(
    columns={"Nota": "Média"},
    inplace=True
)

medias_alunos

## 31. Estatísticas gerais

In [ ]:
media_geral = medias_alunos["Média"].mean()

# Critério do slide:
# aprovado >= 7
aprovados = (medias_alunos["Média"] >= 7).sum()

# No slide aparece também reprovado < 5 para esta estatística específica.
reprovados_abaixo_5 = (medias_alunos["Média"] < 5).sum()

print("Média geral:", round(media_geral, 2))
print("Aprovados (>= 7):", aprovados)
print("Reprovados (< 5):", reprovados_abaixo_5)

## Observação sobre o critério

Nos slides aparecem dois recortes:

- em uma parte: **Aprovado ≥ 7** e **Reprovado < 5**;
- no gráfico de pizza posterior: **Aprovado ≥ 7** e **Reprovado < 7**.

No notebook mantemos os dois exemplos separadamente para refletir o conteúdo apresentado.

## 32. Frequência por aluno

In [ ]:
# Quantidade total de registros/aulas por aluno.
total_aulas = df.groupby("Aluno").size()

# Quantidade de presenças.
total_presencas = (
    df[df["Presenca"] == "Sim"]
    .groupby("Aluno")
    .size()
)

frequencia = pd.DataFrame({
    "Aulas": total_aulas,
    "Presencas": total_presencas
})

frequencia["Frequencia (%)"] = (
    frequencia["Presencas"] /
    frequencia["Aulas"] *
    100
)

frequencia

## 33. Juntando média e frequência

In [ ]:
resultado = medias_alunos.merge(
    frequencia.reset_index(),
    on="Aluno"
)

resultado

## 34. Histograma das médias

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))

resultado["Média"].plot.hist(
    bins=5
)

plt.title("Distribuição das Médias dos Alunos")
plt.xlabel("Média")
plt.ylabel("Quantidade de Alunos")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

## 35. Boxplot das médias

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))

plt.boxplot(resultado["Média"])

plt.title("Boxplot das Médias")
plt.ylabel("Média")
plt.show()

## 36. Gráfico de pizza do desempenho

In [ ]:
import matplotlib.pyplot as plt

categorias = [
    "Aprovados (≥ 7)",
    "Reprovados (< 7)"
]

valores = [
    (resultado["Média"] >= 7).sum(),
    (resultado["Média"] < 7).sum()
]

plt.figure(figsize=(6, 6))

plt.pie(
    valores,
    labels=categorias,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Desempenho Geral dos Alunos")
plt.axis("equal")
plt.tight_layout()
plt.show()

## 37. Média por curso

In [ ]:
# Primeiro associamos cada aluno ao seu curso.
curso_aluno = (
    df[["Aluno", "Curso"]]
    .drop_duplicates()
)

resultado_curso = resultado.merge(
    curso_aluno,
    on="Aluno"
)

media_por_curso = (
    resultado_curso
    .groupby("Curso")["Média"]
    .mean()
    .sort_values(ascending=False)
)

media_por_curso

## 38. Gráfico de médias por curso

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))

media_por_curso.plot(
    kind="bar"
)

plt.title("Média dos Alunos por Curso")
plt.xlabel("Curso")
plt.ylabel("Média")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 39. Dashboard simples com pandas + matplotlib

Vamos colocar alguns indicadores antes dos gráficos.

In [ ]:
total_alunos = resultado["Aluno"].nunique()
media_turma = resultado["Média"].mean()
taxa_aprovacao = (
    (resultado["Média"] >= 7).mean() * 100
)
frequencia_media = resultado["Frequencia (%)"].mean()

print("=== INDICADORES ===")
print("Alunos:", total_alunos)
print("Média da turma:", round(media_turma, 2))
print("Taxa de aprovação:", round(taxa_aprovacao, 1), "%")
print("Frequência média:", round(frequencia_media, 1), "%")

### Dashboard — distribuição das médias

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.hist(resultado["Média"], bins=5)
plt.title("Dashboard — Distribuição das Médias")
plt.xlabel("Média")
plt.ylabel("Quantidade")
plt.grid(axis="y")
plt.show()

### Dashboard — frequência por aluno

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))

plt.bar(
    resultado["Aluno"],
    resultado["Frequencia (%)"]
)

plt.title("Dashboard — Frequência por Aluno")
plt.xlabel("Aluno")
plt.ylabel("Frequência (%)")
plt.ylim(0, 100)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 40. Exportando resultados para Excel

Podemos gerar um arquivo com várias abas contendo diferentes análises.

In [ ]:
import pandas as pd

with pd.ExcelWriter(
    "relatorio_desempenho.xlsx",
    engine="openpyxl"
) as writer:

    # Dados originais.
    df.to_excel(
        writer,
        sheet_name="Dados",
        index=False
    )

    # Resultado por aluno.
    resultado.to_excel(
        writer,
        sheet_name="Resultado por Aluno",
        index=False
    )

    # Média por curso.
    media_por_curso.reset_index().to_excel(
        writer,
        sheet_name="Media por Curso",
        index=False
    )

print("relatorio_desempenho.xlsx criado.")

## 41. Conferindo as abas exportadas

In [ ]:
abas = pd.read_excel(
    "relatorio_desempenho.xlsx",
    sheet_name=None
)

print(abas.keys())

# 42. CSV com ponto e vírgula

Em alguns contextos, especialmente por configurações regionais, CSV pode utilizar `;` como delimitador.

In [ ]:
import pandas as pd

dados = pd.DataFrame({
    "Nome": ["Ana", "Bruno", "Carla"],
    "Cidade": ["Fortaleza", "Quixadá", "Sobral"],
    "Nota": [8.5, 7.0, 9.2]
})

dados.to_csv(
    "dados_ponto_virgula.csv",
    sep=";",
    index=False,
    encoding="utf-8"
)

print("Arquivo criado.")

In [ ]:
# Precisamos informar o separador correto durante a leitura.
df_ponto_virgula = pd.read_csv(
    "dados_ponto_virgula.csv",
    sep=";"
)

df_ponto_virgula

# 43. Problema: separador incorreto

Veja o que acontece se tentarmos ler um arquivo separado por `;` usando a configuração padrão.

In [ ]:
df_incorreto = pd.read_csv(
    "dados_ponto_virgula.csv"
)

df_incorreto

A tabela aparece com uma única coluna porque o `pandas` procurou vírgulas, mas o arquivo usa ponto e vírgula.

# 44. CSV e codificação

UTF-8 é uma boa escolha para novos arquivos.

Vamos criar um CSV contendo acentuação.

In [ ]:
dados_acentuados = pd.DataFrame({
    "Nome": ["João", "José", "Márcia"],
    "Cidade": ["Quixadá", "Fortaleza", "São Paulo"]
})

dados_acentuados.to_csv(
    "acentuacao.csv",
    index=False,
    encoding="utf-8"
)

print("acentuacao.csv criado.")

In [ ]:
df_acentuacao = pd.read_csv(
    "acentuacao.csv",
    encoding="utf-8"
)

df_acentuacao

# 45. Desempenho e arquivos CSV grandes

Para arquivos muito grandes, podemos usar `chunksize`.

Em vez de carregar tudo na memória, o pandas fornece partes do arquivo.

In [ ]:
import pandas as pd

# Vamos gerar um CSV maior para demonstração.
with open("grande.csv", "w", encoding="utf-8") as arquivo:
    arquivo.write("id,valor,categoria\n")

    for i in range(1, 100001):
        categoria = "A" if i % 2 == 0 else "B"
        arquivo.write(f"{i},{i * 2},{categoria}\n")

print("grande.csv criado.")

## 46. Leitura em chunks

In [ ]:
import pandas as pd

contador = 0

# Cada chunk terá no máximo 20.000 linhas.
for chunk in pd.read_csv(
    "grande.csv",
    chunksize=20000
):
    print("Chunk:", chunk.shape)

    contador += len(chunk)

print("\nTotal processado:", contador)

## 47. Processando uma estatística por chunks

In [ ]:
soma = 0
quantidade = 0

for chunk in pd.read_csv(
    "grande.csv",
    chunksize=20000
):
    soma += chunk["valor"].sum()
    quantidade += chunk["valor"].count()

media = soma / quantidade

print("Média da coluna valor:", media)

## 48. Carregando apenas colunas necessárias

Outra otimização é evitar carregar colunas que não serão utilizadas.

In [ ]:
df_reduzido = pd.read_csv(
    "grande.csv",
    usecols=["id", "categoria"]
)

df_reduzido.head()

# 49. Atividade 1 — Cadastro de notas em CSV

Crie um programa que:

1. solicite nome, nota 1 e nota 2;
2. grave em `notas_alunos.csv`;
3. permita cadastrar vários alunos;
4. leia os dados com pandas;
5. calcule a média;
6. classifique cada aluno em Aprovado/Reprovado.

In [ ]:
# SOLUÇÃO DE REFERÊNCIA

import csv
import os
import pandas as pd

ARQUIVO = "notas_alunos.csv"

# Verificamos se o arquivo já existe para decidir
# se precisamos escrever o cabeçalho.
arquivo_existe = os.path.exists(ARQUIVO)

with open(
    ARQUIVO,
    "a",
    newline="",
    encoding="utf-8"
) as arquivo:

    writer = csv.writer(arquivo)

    if not arquivo_existe:
        writer.writerow(["Nome", "Nota1", "Nota2"])

    while True:

        nome = input("Nome do aluno ou 'sair': ").strip()

        if nome.lower() == "sair":
            break

        try:
            nota1 = float(input("Nota 1: "))
            nota2 = float(input("Nota 2: "))

        except ValueError:
            print("Digite notas numéricas.")
            continue

        writer.writerow([nome, nota1, nota2])

print("Dados gravados.")

# -------------------------------------------------------
# Análise
# -------------------------------------------------------

df = pd.read_csv(ARQUIVO)

df["Media"] = (
    df["Nota1"] + df["Nota2"]
) / 2

df["Status"] = df["Media"].apply(
    lambda media: "Aprovado"
    if media >= 7
    else "Reprovado"
)

df

# 50. Atividade 2 — Relatório Acadêmico

A partir de `frequencia_notas.csv`:

1. calcule a média de cada aluno;
2. calcule sua frequência;
3. indique Aprovado/Reprovado;
4. gere um histograma;
5. gere um gráfico de desempenho;
6. exporte os resultados para `relatorio_final.xlsx`.

In [ ]:
# SOLUÇÃO DE REFERÊNCIA

import pandas as pd

dados = pd.read_csv("frequencia_notas.csv")

# Mantém apenas registros válidos para nota.
presentes = dados[
    (dados["Presenca"] == "Sim") &
    dados["Nota"].notna()
]

# Média por aluno.
medias = (
    presentes
    .groupby("Aluno", as_index=False)["Nota"]
    .mean()
    .rename(columns={"Nota": "Media"})
)

# Total de aulas.
aulas = (
    dados
    .groupby("Aluno")
    .size()
    .rename("Aulas")
)

# Total de presenças.
presencas = (
    dados[dados["Presenca"] == "Sim"]
    .groupby("Aluno")
    .size()
    .rename("Presencas")
)

freq = pd.concat(
    [aulas, presencas],
    axis=1
).fillna(0)

freq["Frequencia"] = (
    freq["Presencas"] /
    freq["Aulas"] *
    100
)

resultado_final = medias.merge(
    freq.reset_index(),
    on="Aluno"
)

resultado_final["Status"] = (
    resultado_final["Media"]
    .apply(
        lambda media:
        "Aprovado"
        if media >= 7
        else "Reprovado"
    )
)

resultado_final

In [ ]:
# Histograma
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.hist(resultado_final["Media"], bins=5)
plt.title("Distribuição das Médias")
plt.xlabel("Média")
plt.ylabel("Alunos")
plt.show()

In [ ]:
# Distribuição de status
status = resultado_final["Status"].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(
    status,
    labels=status.index,
    autopct="%1.1f%%"
)
plt.title("Aprovados e Reprovados")
plt.axis("equal")
plt.show()

In [ ]:
# Exportação final
with pd.ExcelWriter(
    "relatorio_final.xlsx",
    engine="openpyxl"
) as writer:

    dados.to_excel(
        writer,
        sheet_name="Dados Originais",
        index=False
    )

    resultado_final.to_excel(
        writer,
        sheet_name="Resultado Final",
        index=False
    )

print("relatorio_final.xlsx criado.")

# Fechamento

Nesta aula percorremos a seguinte evolução:

```text
Dados tabulares
     ↓
CSV
     ↓
RFC 4180
     ↓
TSV
     ↓
pandas
     ↓
XLSX
     ↓
openpyxl
     ↓
Análise
     ↓
Visualização
     ↓
Exportação
     ↓
Processamento de arquivos maiores
```

## Pontos principais

- CSV e TSV são formatos textuais;
- CSV exige cuidados com vírgulas, aspas e quebras de linha;
- a biblioteca `csv` evita manipulação manual incorreta;
- `pandas` facilita leitura, filtragem, agrupamento e análise;
- XLSX permite múltiplas abas e recursos de planilha;
- `openpyxl` permite manipulação direta das células;
- histogramas mostram distribuições;
- boxplots mostram quartis, dispersão e possíveis outliers;
- gráficos de pizza representam proporções;
- `groupby()` é fundamental para análises agregadas;
- `to_excel()` permite exportar resultados;
- `chunksize` permite processar CSVs maiores sem carregar tudo de uma vez.